# Day 007 — Binary Tree · Heap Sort · Exponential Search
**Date:** 2026-08-17  |  **Difficulty:** Intermediate  |  **Series:** Daily DSA

---

## What You Will Learn Today
| Topic | Concept | Time Complexity |
|-------|---------|----------------|
| Data Structure | **Binary Tree** | Insert O(n), Search O(n) general; O(log n) balanced |
| Sorting | **Heap Sort** | O(n log n) always, in-place, not stable |
| Searching | **Exponential Search** | O(log n) — doubles range until target fits, then Binary Search |

> **Series recap**
> - Day 001: Array · Bubble Sort · Linear Search
> - Day 002: Singly Linked List · Selection Sort · Jump Search
> - Day 003: Doubly Linked List · Insertion Sort · Binary Search
> - Day 004: Stack · Shell Sort · Sentinel Search
> - Day 005: Queue · Merge Sort · Interpolation Search
> - Day 006: Circular Queue · Quick Sort · Fibonacci Search
> - **Today**: Binary Tree (general) · Heap Sort (max-heap in-place) · Exponential Search

> Run each cell top-to-bottom with **Shift+Enter** to follow along interactively.

---
## PART 1 — Data Structure: Binary Tree

A **Binary Tree** is a tree where every node has **at most two children**: left and right.
Unlike a Binary Search Tree (Day 008), a general binary tree has **no ordering constraint**.

### Visual
```
          1          ← root (level 0)
        /   \
       2     3       ← level 1
      / \     \
     4   5     6    ← level 2  (leaves: 4, 5, 6)
```

### Tree traversals
| Traversal | Order | Use case |
|-----------|-------|----------|
| Pre-order | root → left → right | Copy/serialize a tree |
| In-order | left → root → right | BST sorted output |
| Post-order | left → right → root | Delete tree, evaluate expressions |
| Level-order (BFS) | level by level | Shortest path, serialization |

### Key properties
- **Height** h: longest root-to-leaf path. Full balanced tree: h = ⌊log₂ n⌋
- **Perfect** binary tree: all levels full — exactly 2ʰ⁺¹ - 1 nodes
- **Complete** binary tree: all levels full except last, filled left-to-right (heap property)
- **Degenerate** tree: every node has one child — behaves like a linked list, O(n) ops

In [ ]:
# ─── Binary Tree Implementation ──────────────────────────────────────────────

from collections import deque

class TreeNode:
    __slots__ = ('val', 'left', 'right')
    def __init__(self, val):
        self.val   = val
        self.left  = None
        self.right = None
    def __repr__(self): return f'TreeNode({self.val})'


class BinaryTree:
    """
    General (non-BST) Binary Tree.
    Insertion uses level-order BFS to fill the tree left-to-right,
    producing a complete binary tree — the same shape a heap uses.
    """

    def __init__(self):
        self.root = None

    def insert(self, val):
        """Level-order insert — fills left before right at each level. O(n)."""
        node = TreeNode(val)
        if self.root is None:
            self.root = node
            return
        q = deque([self.root])
        while q:
            curr = q.popleft()
            if curr.left is None:
                curr.left = node
                return
            q.append(curr.left)
            if curr.right is None:
                curr.right = node
                return
            q.append(curr.right)

    def pre_order(self, node=None, _first=True):
        """Root → Left → Right — O(n)."""
        if _first: node = self.root
        if node is None: return []
        return [node.val] + self.pre_order(node.left, False) + self.pre_order(node.right, False)

    def in_order(self, node=None, _first=True):
        """Left → Root → Right — O(n)."""
        if _first: node = self.root
        if node is None: return []
        return self.in_order(node.left, False) + [node.val] + self.in_order(node.right, False)

    def post_order(self, node=None, _first=True):
        """Left → Right → Root — O(n)."""
        if _first: node = self.root
        if node is None: return []
        return self.post_order(node.left, False) + self.post_order(node.right, False) + [node.val]

    def level_order(self):
        """BFS level-by-level — returns list of levels. O(n)."""
        if self.root is None: return []
        result, q = [], deque([self.root])
        while q:
            level = []
            for _ in range(len(q)):
                node = q.popleft()
                level.append(node.val)
                if node.left:  q.append(node.left)
                if node.right: q.append(node.right)
            result.append(level)
        return result

    def height(self, node=None, _first=True):
        """Height = longest root-to-leaf path (0 for single node). O(n)."""
        if _first: node = self.root
        if node is None: return -1
        return 1 + max(self.height(node.left, False), self.height(node.right, False))

    def count(self):
        """Total number of nodes — O(n)."""
        def _count(node):
            if node is None: return 0
            return 1 + _count(node.left) + _count(node.right)
        return _count(self.root)

    def __repr__(self):
        levels = self.level_order()
        return 'BinaryTree(\n' + '\n'.join(f'  L{i}: {lvl}' for i, lvl in enumerate(levels)) + '\n)'


bt = BinaryTree()
print('--- Inserting 1..7 (level-order) ---')
for v in range(1, 8):
    bt.insert(v)

print(bt)
print(f'Height     : {bt.height()}')
print(f'Node count : {bt.count()}')
print()
print(f'Pre-order  (root→L→R): {bt.pre_order()}')
print(f'In-order   (L→root→R): {bt.in_order()}')
print(f'Post-order (L→R→root): {bt.post_order()}')
print(f'Level-order (BFS)    : {bt.level_order()}')

In [ ]:
# ─── Binary Tree: search + practical applications ────────────────────────────────────

def search(root, target):
    """BFS search — O(n) for general tree, O(log n) for balanced BST."""
    if root is None: return None
    q = deque([root])
    while q:
        node = q.popleft()
        if node.val == target: return node
        if node.left:  q.append(node.left)
        if node.right: q.append(node.right)
    return None


def mirror(root):
    """Return a mirror-image (flip left/right at every node) — O(n)."""
    if root is None: return None
    node = TreeNode(root.val)
    node.left  = mirror(root.right)
    node.right = mirror(root.left)
    return node


def is_symmetric(root):
    """True if the tree is a mirror of itself — O(n)."""
    def _check(l, r):
        if l is None and r is None: return True
        if l is None or r is None:  return False
        return l.val == r.val and _check(l.left, r.right) and _check(l.right, r.left)
    return root is None or _check(root.left, root.right)


print('--- Search ---')
for target in [4, 7, 99]:
    result = search(bt.root, target)
    print(f'  search({target}) → {result}')

print('\n--- Mirror ---')
mirrored_root = mirror(bt.root)
m = BinaryTree(); m.root = mirrored_root
print(f'Original level-order : {bt.level_order()}')
print(f'Mirrored level-order : {m.level_order()}')

print('\n--- Symmetry ---')
sym = BinaryTree()
for v in [1, 2, 2, 3, 4, 4, 3]:
    sym.insert(v)
print(f'Tree {sym.level_order()} symmetric? {is_symmetric(sym.root)}')
print(f'Tree {bt.level_order()}  symmetric? {is_symmetric(bt.root)}')

---
## PART 2 — Sorting Algorithm: Heap Sort

**Heap Sort** uses a **max-heap** to sort in place:
1. **Build** a max-heap from the array — O(n)
2. Repeatedly **extract the max** (swap root with last, shrink heap, sift down) — O(n log n)

### Max-heap property
Every parent ≥ both children. The array representation maps:
```
parent(i)       = (i - 1) // 2
left_child(i)   = 2 * i + 1
right_child(i)  = 2 * i + 2
```

### Visual: array [4, 10, 3, 5, 1] → max-heap → sorted
```
Build max-heap:        [10, 5, 3, 4, 1]
Extract 10 → [1,5,3,4|10] → sift → [5,4,3,1|10]
Extract  5 → [1,4,3|5,10] → sift → [4,1,3|5,10]
Extract  4 → [3,1|4,5,10] → sift → [3,1|4,5,10]
Extract  3 → [1|3,4,5,10]           done
Sorted:                [1, 3, 4, 5, 10]
```

### Complexity
| Phase | Time | Space |
|-------|------|-------|
| Build heap (heapify) | O(n) | O(1) |
| Extract all | O(n log n) | O(1) |
| Total | **O(n log n)** always | **O(1)** in-place |

**Not stable** — equal elements may change relative order. Use Merge Sort when stability matters.

In [ ]:
# ─── Heap Sort ────────────────────────────────────────────────────────────────────────

def _sift_down(arr, n, i, counters, verbose):
    """
    Restore max-heap property at index i for a heap of size n.
    Moves arr[i] down until it is >= both children.
    """
    largest = i
    left    = 2 * i + 1
    right   = 2 * i + 2

    if left < n:
        counters['comparisons'] += 1
        if arr[left] > arr[largest]:
            largest = left
    if right < n:
        counters['comparisons'] += 1
        if arr[right] > arr[largest]:
            largest = right

    if largest != i:
        arr[i], arr[largest] = arr[largest], arr[i]
        counters['swaps'] += 1
        if verbose:
            print(f'    sift_down: swap index {i} ↔ {largest} → {arr[:n]}')
        _sift_down(arr, n, largest, counters, verbose)


def heap_sort(arr, verbose=False):
    """
    In-place Heap Sort — returns (sorted_list, swaps, comparisons).
    Phase 1: build max-heap in O(n) by sifting down from last non-leaf.
    Phase 2: extract max n-1 times in O(n log n).
    """
    a = arr[:]
    n = len(a)
    counters = {'swaps': 0, 'comparisons': 0}

    if verbose: print('Phase 1: Build max-heap')
    for i in range(n // 2 - 1, -1, -1):
        _sift_down(a, n, i, counters, verbose)
    if verbose: print(f'  max-heap: {a}\n')

    if verbose: print('Phase 2: Extract max')
    for end in range(n - 1, 0, -1):
        a[0], a[end] = a[end], a[0]
        counters['swaps'] += 1
        if verbose:
            print(f'  extract {a[end]} → sorted region: {a[end:]}, heap: {a[:end]}')
        _sift_down(a, end, 0, counters, verbose)

    return a, counters['swaps'], counters['comparisons']


sample = [4, 10, 3, 5, 1]
print(f'Input: {sample}\n')
sorted_arr, swaps, comps = heap_sort(sample, verbose=True)
print(f'\nResult: {sorted_arr}')
print(f'Swaps: {swaps}  |  Comparisons: {comps}')

In [ ]:
# ─── Heap Sort test suite ─────────────────────────────────────────────────────────

test_cases = [
    ([4, 10, 3, 5, 1],         'random small'),
    ([1, 2, 3, 4, 5],          'already sorted'),
    ([5, 4, 3, 2, 1],          'reverse sorted'),
    ([42],                     'single element'),
    ([],                       'empty list'),
    ([3, 3, 1, 1, 2, 2],       'with duplicates'),
    ([-5, 0, 3, -2, 8, -1],    'with negatives'),
    (list(range(20, 0, -1)),   'reverse 1-20'),
    ([7] * 8,                  'all identical'),
]

print(f'{"Input":<34} {"Sorted":<34} {"Swaps":>5} {"Cmps":>5}  Case')
print('-' * 95)
for data, label in test_cases:
    result, sw, cm = heap_sort(data)
    assert result == sorted(data), f'FAIL on {label}'
    print(f'{str(data):<34} {str(result):<34} {sw:>5} {cm:>5}  {label}')

print('\nAll tests passed ✓')

---
## PART 3 — Searching Algorithm: Exponential Search

**Exponential Search** finds a range where the target could exist by doubling the
index (1 → 2 → 4 → 8 → …), then performs **Binary Search** within that range.

### How it works
```
Array: [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21]  target = 15

Step 1 — exponential phase (double i until arr[i] >= target or end):
  i=1: arr[1]=3  < 15 → double → i=2
  i=2: arr[2]=5  < 15 → double → i=4
  i=4: arr[4]=9  < 15 → double → i=8
  i=8: arr[8]=17 >= 15 → STOP

Step 2 — Binary Search in arr[i//2 .. min(i, n-1)] = arr[4..8]
  lo=4, hi=8 → mid=6, arr[6]=13 < 15 → lo=7
  lo=7, hi=8 → mid=7, arr[7]=15 == 15 → FOUND at index 7
```

### Complexity
| Phase | Time |
|-------|------|
| Exponential phase | O(log p) where p = target's position |
| Binary Search phase | O(log p) |
| Total | **O(log p)** — faster than Binary Search when target is near the start |

### When to prefer Exponential over Binary Search?
- **Unbounded / infinite arrays** — you don't know n upfront
- Target is **near the beginning** — exponential phase is very fast
- Binary Search still wins when target is near the end

In [ ]:
# ─── Exponential Search ─────────────────────────────────────────────────────────────

def _binary_search(arr, target, lo, hi, verbose):
    """Standard iterative Binary Search within arr[lo..hi]."""
    step = 0
    while lo <= hi:
        mid = lo + (hi - lo) // 2
        step += 1
        if verbose:
            print(f'    BS step {step}: lo={lo}, hi={hi}, mid={mid}, arr[mid]={arr[mid]}')
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1


def exponential_search(arr, target, verbose=False):
    """
    Exponential Search on a sorted array.
    Returns the index of target, or -1 if not found.
    """
    n = len(arr)
    if n == 0:
        return -1
    if arr[0] == target:
        if verbose: print(f'  Found at index 0 immediately')
        return 0

    i = 1
    exp_steps = 0
    while i < n and arr[i] <= target:
        exp_steps += 1
        if verbose:
            print(f'  Exp step {exp_steps}: i={i}, arr[i]={arr[i]} <= {target} → double to {i*2}')
        i *= 2

    lo = i // 2
    hi = min(i, n - 1)
    if verbose:
        print(f'  Range found: [{lo}, {hi}] — running Binary Search')

    return _binary_search(arr, target, lo, hi, verbose)


data = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21]
print(f'Array: {data}\n')

print('Search for 15 (present):')
idx = exponential_search(data, 15, verbose=True)
print(f'Result: index {idx} → value {data[idx] if idx != -1 else "not found"}\n')

print('Search for 3 (near start — exponential fast):')
idx = exponential_search(data, 3, verbose=True)
print(f'Result: index {idx} → value {data[idx] if idx != -1 else "not found"}\n')

print('Search for 8 (absent):')
idx = exponential_search(data, 8, verbose=True)
print(f'Result: {idx} (not found)')

In [ ]:
# ─── Exponential Search test suite ───────────────────────────────────────────────────

sorted_data = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21]

search_tests = [
    (sorted_data, 1,   'first element (index 0)'),
    (sorted_data, 21,  'last element'),
    (sorted_data, 11,  'middle element'),
    (sorted_data, 3,   'near start — exp fast'),
    (sorted_data, 8,   'not found — between elements'),
    (sorted_data, 0,   'not found — below range'),
    (sorted_data, 22,  'not found — above range'),
    ([42],         42, 'single element — found'),
    ([42],         7,  'single element — not found'),
    ([],           5,  'empty array'),
    (list(range(0, 100, 2)), 64, 'large even array, target=64'),
]

print(f'{"Array[:6]":<28} {"Target":>7}  {"Result":<14} Case')
print('-' * 72)
for arr, target, label in search_tests:
    result   = exponential_search(arr, target)
    found    = f'index {result}' if result != -1 else 'not found'
    expected = arr.index(target) if target in arr else -1
    ok = '✓' if result == expected else f'✗ (expected {expected})'
    preview = str(arr[:6]) + ('...' if len(arr) > 6 else '')
    print(f'{preview:<28} {str(target):>7}  {found:<14} {ok}  {label}')

print('\nAll tests passed ✓')

---
## PART 4 — End-to-End: File System Directory Tree

Model a simplified file system:
1. **Binary Tree** stores directory entries (each node = a file/folder)
2. **Heap Sort** orders entries by file size
3. **Exponential Search** locates a file by size in the sorted list

In [ ]:
# ─── File System simulation ─────────────────────────────────────────────────────────

files = [
    {'name': 'readme.txt',   'size_kb': 4},
    {'name': 'video.mp4',    'size_kb': 512},
    {'name': 'notes.md',     'size_kb': 12},
    {'name': 'photo.jpg',    'size_kb': 128},
    {'name': 'script.py',    'size_kb': 8},
    {'name': 'archive.zip',  'size_kb': 256},
    {'name': 'config.json',  'size_kb': 2},
]

print('=== Step 1: Build Binary Tree from directory entries ===')
dir_tree = BinaryTree()
for f in files:
    dir_tree.insert(f['name'])
print(dir_tree)

print('Traversals:')
print(f'  Pre-order  : {dir_tree.pre_order()}')
print(f'  Level-order: {dir_tree.level_order()}')
print(f'  Height     : {dir_tree.height()}')

print('\n=== Step 2: Heap Sort files by size ===')
sizes = [f['size_kb'] for f in files]
print(f'Unsorted sizes (KB): {sizes}')
sorted_sizes, swaps, comps = heap_sort(sizes)
print(f'Sorted   sizes (KB): {sorted_sizes}  ({swaps} swaps, {comps} comparisons)')

size_to_name = {f['size_kb']: f['name'] for f in files}
print('\nSorted directory (smallest to largest):')
for sz in sorted_sizes:
    print(f'  {sz:>6} KB  {size_to_name[sz]}')

print('\n=== Step 3: Exponential Search for file of size 128 KB ===')
target_size = 128
pos = exponential_search(sorted_sizes, target_size, verbose=True)
if pos != -1:
    matched = size_to_name[sorted_sizes[pos]]
    print(f'\nFound {target_size} KB at sorted index {pos} → "{matched}"')
else:
    print(f'\n{target_size} KB not found in directory')

---
## Complexity Cheat Sheet

```
Binary Tree insert (level-order)    O(n)
Binary Tree traversals              O(n)  — all four variants
Binary Tree height / count          O(n)
Binary Tree BFS search              O(n)  — no ordering guarantee

Heap Sort build-heap phase          O(n)  — heapify bottom-up
Heap Sort extract phase             O(n log n)
Heap Sort total                     O(n log n), O(1) space, NOT stable

Exponential Search (exp phase)      O(log p)  — p = target's position
Exponential Search (binary phase)   O(log p)
Exponential Search total            O(log p)  — beats Binary O(log n) when p << n
```

**Key insight:** Heap Sort guarantees O(n log n) in ALL cases (unlike Quick Sort's O(n²) worst)
and uses O(1) extra space (unlike Merge Sort's O(n)). The trade-off: it is not stable and
has worse cache behaviour than Quick Sort due to non-sequential memory access in the heap.

**Tomorrow — Day 008:** Binary Search Tree · Counting Sort · Ternary Search